In [9]:
import json
import os
import re
import pandas as pd
import numpy as np
import string
from collections import Counter
import textstat
import spacy
import pprint

In [2]:
# Load data
folder_path = "../results/base/base_none_5/"
data = []

for root, dirs, files in os.walk(folder_path):
    for filename in files:
        if filename.endswith(".json"):
            file_path = os.path.join(root, filename)
            try:
                with open(file_path, "r") as f:
                    entry = json.load(f)
                    data.append(entry)
            except json.JSONDecodeError as e:
                print(f"Skipped invalid JSON: {file_path} ({e})")

print(f"Loaded {len(data)} valid JSON result files from '{folder_path}' and its subfolders.")

Loaded 4218 valid JSON result files from '../results/base_none_5/' and its subfolders.


In [5]:
nlp = spacy.load("en_core_web_sm")

MODEL_FAMILIES = ["gemini", "claude", "gpt", "llama", "qwen", "deepseek", "mistral"]

def avg_words_per_sentence(utterances):
    sentences = [sent for utt in utterances for sent in nlp(utt).sents]
    if not sentences:
        return 0
    return np.mean([len(sent.text.split()) for sent in sentences])

def bullet_points_per_utterance(utterances):
    bullets = [re.findall(r"^\s*[-*•]\s", utt, flags=re.M) for utt in utterances]
    return np.mean([len(b) for b in bullets]) if bullets else 0

def headings_per_utterance(utterances):
    heading_regex = r"^\s*(#+|[A-Z][A-Z ]{3,})"
    headings = [re.findall(heading_regex, utt, flags=re.M) for utt in utterances]
    return np.mean([len(h) for h in headings]) if headings else 0

def sentence_complexity(utterances):
    all_text = " ".join(utterances)
    return textstat.flesch_kincaid_grade(all_text)

def lexical_diversity(utterances):
    tokens = [tok.lower() for utt in utterances for tok in utt.split() if tok.isalpha()]
    if not tokens:
        return 0
    return len(set(tokens)) / len(tokens)

def formality_score(utterances):
    # Rough proxy: proportion of modal verbs and absence of contractions
    formality_hits = 0
    total_tokens = 0
    modal_verbs = {"would", "could", "should", "may", "might", "shall"}
    contractions = re.compile(r"\b\w+'\w+\b")
    for utt in utterances:
        tokens = utt.lower().split()
        total_tokens += len(tokens)
        formality_hits += sum(1 for t in tokens if t in modal_verbs)
        formality_hits -= len(re.findall(contractions, utt))
    return formality_hits / total_tokens if total_tokens else 0

def punctuation_density(utterances):
    all_text = "".join(utterances)
    punct_count = sum(1 for c in all_text if c in string.punctuation)
    return punct_count / len(all_text) if all_text else 0

def output_structure_consistency(utterances):
    # Fraction of utterances that follow bullet or heading structure
    structured = sum(
        1 for utt in utterances if re.search(r"^\s*[-*•#]", utt, re.M)
    )
    return structured / len(utterances) if utterances else 0

def extract_features_from_run(run):
    utterances = run["utterances"]
    features = {
        "run_id": run["run_id"],
        "avg_words_per_sentence": avg_words_per_sentence(utterances),
        "bullet_points_per_utterance": bullet_points_per_utterance(utterances),
        "headings_per_utterance": headings_per_utterance(utterances),
        "sentence_complexity": sentence_complexity(utterances),
        "lexical_diversity": lexical_diversity(utterances),
        "formality_score": formality_score(utterances),
        "punctuation_density": punctuation_density(utterances),
        "output_structure_consistency": output_structure_consistency(utterances),
    }
    # Add one-hot labels for each model family
    for model in MODEL_FAMILIES:
        features[f"is_{model}"] = int(model in run["model_names"])
    return features

In [6]:
runs = []
for entry in data:
    for agent_id in [0, 1]:
        role = f"agent_{agent_id}"
        model = entry[f"model_general_name_{agent_id}"]
        utterances = [row["content"] for row in entry["dialogue"] if row["role"] == role]
        runs.append({
            "run_id": len(runs),
            "utterances": utterances,
            "model_names": [model]
        })
    
feature_rows = [extract_features_from_run(run) for run in runs]
df = pd.DataFrame(feature_rows)
df.to_csv("figures_and_files/feature_data.csv", index=False)

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score
import pandas as pd

# Load your feature data
df = pd.read_csv("figures_and_files/feature_data.csv")  # assume this has all features + labels

# List of model families you want to classify
model_families = ["gemini", "claude", "gpt", "llama", "qwen", "deepseek", "mistral"]

# Results dictionary
results = {}

# Loop through each model and train classifier
for model in model_families:
    label_col = f"is_{model}"
    
    # Features and labels
    X = df.drop(columns=[f"is_{m}" for m in model_families] + ['run_id'])  # drop labels + IDs
    y = df[label_col]
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
    
    # Logistic Regression classifier
    clf = LogisticRegression(max_iter=1000, class_weight='balanced')
    clf.fit(X_train, y_train)
    
    # Evaluation
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    
    results[model] = {
        "report": classification_report(y_test, y_pred, output_dict=True),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "coefficients": dict(zip(X.columns, clf.coef_[0]))
    }

In [15]:
pprint.pprint(results["qwen"])

{'coefficients': {'avg_words_per_sentence': np.float64(-0.01641052386142933),
                  'bullet_points_per_utterance': np.float64(0.12800507417776774),
                  'formality_score': np.float64(10.52421793955302),
                  'headings_per_utterance': np.float64(0.07771181195145725),
                  'lexical_diversity': np.float64(-1.634522877560924),
                  'output_structure_consistency': np.float64(0.3959533053978467),
                  'punctuation_density': np.float64(7.566024146387244),
                  'sentence_complexity': np.float64(-0.32973722471248823)},
 'report': {'0': {'f1-score': 0.8400866068666872,
                  'precision': 0.9536516853932584,
                  'recall': 0.7506909894969597,
                  'support': 1809.0},
            '1': {'f1-score': 0.4751269035532995,
                  'precision': 0.3416058394160584,
                  'recall': 0.78,
                  'support': 300.0},
            'accuracy': 0.754860123